# Download and Import Package

In [1]:
# !pip install -U torch torchvision
!pip install -U bitsandbytes transformers pillow accelerate

In [2]:
!pip install pymupdf==1.26.6 pymupdf-layout==1.26.6 pymupdf4llm==0.2.9

  Using cached pymupdf-1.26.6-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (3.4 kB)
  Using cached pymupdf_layout-1.26.6-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (2.8 kB)
  Using cached pymupdf4llm-0.2.9-py3-none-any.whl.metadata (7.4 kB)
  Using cached onnxruntime-1.24.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (4.9 kB)
Using cached pymupdf-1.26.6-cp310-abi3-manylinux_2_28_x86_64.whl (24.1 MB)
Using cached pymupdf_layout-1.26.6-cp310-abi3-manylinux_2_28_x86_64.whl (12.7 MB)
Using cached pymupdf4llm-0.2.9-py3-none-any.whl (72 kB)
Using cached onnxruntime-1.24.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (17.1 MB)


In [3]:
import json                         # 處理 JSON 格式資料的標準套件 / Standard JSON library
import torch                        # PyTorch 是一個深度學習框架 / PyTorch deep learning framework
from typing import List             # 用來註解 List 型別 / Type annotation for list
from transformers import (
    AutoModelForCausalLM,           # 自動載入因果語言模型 / Auto loader for causal language models
    AutoTokenizer,                  # 自動載入對應的 tokenizer / Auto loader for tokenizer
    BitsAndBytesConfig,             # 用來設定量化模型的參數 / Configuration for model quantization
    AutoProcessor
)
import jieba                        # 中文斷詞工具 / Chinese word segmentation

from google.colab import drive
import pickle
import shutil

import os
from PIL import Image
import requests
from io import BytesIO

from threading import Thread

import pymupdf4llm
import pymupdf.layout
import pathlib
import re

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


# 雲端檔案處理

In [4]:
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/RAG_Data' # 借用已下載完的LLM
if not os.path.exists(BASE_PATH):
    os.makedirs(BASE_PATH)
    print(f"已建立資料夾: {BASE_PATH}")
else:
    print(f"資料夾已存在: {BASE_PATH}")

DOC = "/content/drive/MyDrive/RAG_Data/test/中文測試文件.pdf"
IMAGES_OUTPUT_DIR = "/content/drive/MyDrive/RAG_Data/test/imgs/"
MD_OUTPUT_PATH = "/content/drive/MyDrive/RAG_Data/test/output.md"

Mounted at /content/drive
資料夾已存在: /content/drive/MyDrive/RAG_Data


# Load LLM

In [5]:
# 自動偵測是否支援 bfloat16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

LLM_CACHE_PATH = os.path.join(BASE_PATH, "llm_cache")
if not os.path.exists(LLM_CACHE_PATH):
    os.makedirs(LLM_CACHE_PATH)
    print(f"已建立 LLM 快取資料夾: {LLM_CACHE_PATH}")
else:
    print(f"將使用現有的 LLM 快取: {LLM_CACHE_PATH}")

# 設定量化參數，減少記憶體使用 / Set quantization settings for smaller memory usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 使用 4-bit 權重 / Use 4-bit weights
    bnb_4bit_use_double_quant=True,       # 啟用雙重量化 / Enable double quantization
    bnb_4bit_quant_type="nf4",            # 使用 nf4 量化類型 / Quantization type
    bnb_4bit_compute_dtype=compute_dtype  # 使用 bfloat16 或 float16 進行計算 / Use float16 or float16 for compute
)

# 載入微調好的語言模型（Gemma）/ Load pretrained LLM
llm = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path="unsloth/gemma-3-4b-it",
    quantization_config=bnb_config,
    dtype=compute_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
    cache_dir=LLM_CACHE_PATH
)

processor = AutoProcessor.from_pretrained(
    "unsloth/gemma-3-4b-it",
    trust_remote_code=True,
    use_fast=True,
    cache_dir=LLM_CACHE_PATH
)

將使用現有的 LLM 快取: /content/drive/MyDrive/RAG_Data/llm_cache


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]


# Define Functions


In [18]:
def pdf_to_markdown(doc: str, image_output_dir: str) -> str:
    md_text = pymupdf4llm.to_markdown(
        doc=doc,
        write_images=True,
        image_path=image_output_dir,
        image_format="png",
        dpi=300,
    )
    return md_text


def get_description_for_image(image_path):
    image = Image.open(image_path)
    if image.mode != "RGB":
        image = image.convert("RGB")

    chats = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "請分析並簡短描述圖片內容，回覆內容僅須描述即可，不須包含任何問候或詢問句。"},
            ],
        },
    ]
    text_prompt = processor.apply_chat_template(chats, add_generation_prompt=True)
    inputs = processor(text=text_prompt, images=image, return_tensors="pt").to("cuda")

    generated_ids = llm.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )
    generated_ids = generated_ids[:, inputs['input_ids'].shape[1]:]
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return generated_text

# Main

In [19]:
md_text = pdf_to_markdown(DOC, IMAGES_OUTPUT_DIR)

if not os.path.exists(MD_OUTPUT_PATH):
    open(MD_OUTPUT_PATH, "w").close()
pathlib.Path(MD_OUTPUT_PATH).write_bytes(md_text.encode())

# add captions for images
caption = "This is a caption text for the image."  # caption 放 LLMs 對圖片的描述，像 LlamaIndex 那樣
pattern = r"!\[.*?\]\((.*?)\)"

def process_caption(match):
    image_path = match.group(1)
    caption = get_description_for_image(image_path)
    return f"![Image]({image_path})\n*({caption})*\n\n"

with open(MD_OUTPUT_PATH, "r+", encoding="utf-8") as f:
    content = f.read()
    content = re.sub(pattern, process_caption, content)
    # Print first 2000 characters of the modified content to verify
    print("\n--- Content to be written to MD_OUTPUT_PATH (first 2000 chars) ---")
    print(content[:2000])
    print("-------------------------------------------------------------------")
    f.seek(0)
    f.write(content)
    f.truncate()


--- Content to be written to MD_OUTPUT_PATH (first 2000 chars) ---
中文測試文件標題 | kiol

# 中文測試文件


我是第一段內文，單欄靠左對齊。


我是第二段內文，具有一個縮排。


我是點列說明文字。


  - 我是點列項目 1.

  - 我是點列項目 2.

  - 我是點列項目 3.


我是清單說明文字。


  - 我是清單項目 1.

  - 我是清單項目 2.

  - 我是清單項目 3.


我是表格說明文字。

|Col1|標題一|標題二|標題三|標題四|
|---|---|---|---|---|
|首欄一|||||
|首欄二||我是合併儲存格文字|我是合併儲存格文字|(2, 4)|
|首欄三|(3, 1)||||
|首欄四|||(4, 3)||



1



![Image](/content/drive/MyDrive/RAG_Data/test/imgs/中文測試文件.pdf-0-0.png)
*(圖片展示了兩個決策樹的比較。

左側的樹顯示一個平均值 (x, w) 為 0.2 的樹，其中左分支的樣本比例為 20%，右分支的樣本比例為 80%。

右側的樹顯示一個平均值 (x, w) 為 0.5 的樹，其中左分支的樣本比例為 10%，右分支的樣本比例為 10%，下分支的樣本比例為 56%，以及最右分支的樣本比例為 24%。

兩個樹的平均值都不同，但樹的結構和分支比例有所改變。)*


我是第三段內文，雙欄靠左對齊。



中文測試文件標題 | kiol


我是第四段內文，雙欄靠左對齊。



2



-------------------------------------------------------------------
